# KD-Capacity-Gap: Full Experiment Runner

End-to-end pipeline for the CIFAR-10 knowledge distillation ablation study.

| Step | What happens | Est. time (T4) |
|------|-------------|----------------|
| 1 | Clone repo + install deps | 2 min |
| 2 | Build CIFAR-10 ImageFolder | 3–5 min |
| 3 | Train R50 + R34 teachers (200 ep each) | ~2 h |
| 4 | Run 18 distillation experiments (100 ep each) | ~5 h |
| 5 | Zip & download `runs/` | 1 min |
| 6 | Print mean ± std results table | instant |

> **Recommended:** Colab Pro with A100 cuts total time to ~2–3 h.  
> Run cells top-to-bottom. Each step skips already-completed work, so you can safely re-run after a disconnect.

## 0 — GPU check
Verify a GPU is attached before spending time on setup.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null \
    && echo "" \
    || echo "⚠  No GPU detected — go to Runtime → Change runtime type → T4 GPU."

## 0.5 — Google Drive setup
Mount Drive and define `DRIVE_BASE` — the single variable controlling where all outputs are stored.

In [ ]:
from google.colab import drive
import os, shutil
from pathlib import Path

drive.mount('/content/drive')

# ── Change this to move all outputs to a different Drive folder ─────────
DRIVE_BASE = '/content/drive/MyDrive/kd-capacity-gap'
# ─────────────────────────────────────────────────────────────────────────

Path(DRIVE_BASE).mkdir(parents=True, exist_ok=True)
print(f'Drive base: {DRIVE_BASE}')

## 1 — Clone repo & install requirements

In [ ]:
import os

REPO_URL = "https://github.com/umutonuryasar/kd-capacity-gap.git"

if os.path.basename(os.getcwd()) != "kd-capacity-gap":
    if not os.path.isdir("kd-capacity-gap"):
        !git clone {REPO_URL}
    os.chdir("kd-capacity-gap")
else:
    !git pull --ff-only

print("Working directory:", os.getcwd())

In [ ]:
# torch / torchvision are pre-installed on Colab; remaining deps are lightweight
!pip install -r requirements.txt -q
print("Requirements ready.")

## 2 — Prepare CIFAR-10 data

`train.py` uses `torchvision.datasets.ImageFolder`, so we download CIFAR-10
via torchvision and write each image as a PNG into `data/train/{class}/` and
`data/test/{class}/`. The cell is idempotent — it skips if the images already exist.

In [ ]:
import torchvision
from pathlib import Path
from tqdm.auto import tqdm

CIFAR_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog",      "frog",       "horse", "ship", "truck",
]

def convert_cifar10_to_imagefolder(split: str, train: bool) -> None:
    dest     = Path(f"data/{split}")
    expected = 50_000 if train else 10_000
    existing = len(list(dest.rglob("*.png")))
    if existing >= expected:
        print(f"data/{split}/  already ready ({existing:,} images) — skipping.")
        return

    print(f"Building data/{split}/ …")
    ds = torchvision.datasets.CIFAR10(root="/tmp/cifar10", train=train, download=True)
    for i, (img, label) in enumerate(tqdm(ds, desc=split, unit="img")):
        cls_dir = dest / CIFAR_CLASSES[label]
        cls_dir.mkdir(parents=True, exist_ok=True)
        img.save(cls_dir / f"{i:05d}.png")
    print(f"data/{split}/  ready — {len(ds):,} images.")

convert_cifar10_to_imagefolder("train", train=True)
convert_cifar10_to_imagefolder("test",  train=False)

## 3 — Train teacher models

Trains R50 and R34 from scratch on CIFAR-10 (no KD, pure cross-entropy).
Both use the CIFAR-adapted stem: 3×3 conv (stride=1) + Identity maxpool.

Checkpoints are saved to `checkpoints/teacher_r50.pth` and `checkpoints/teacher_r34.pth`.
**The cell skips a model if its checkpoint already exists.**

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
TEACHER_EPOCHS = 200   # use 200 for publication quality; 50 for a quick smoke-test
TEACHER_SEED   = 0
# ─────────────────────────────────────────────────────────────────────────────

print(f"Teachers: R50 + R34 | {TEACHER_EPOCHS} epochs each | seed {TEACHER_SEED}")
print(f"Estimated time: ~{TEACHER_EPOCHS // 100}–{TEACHER_EPOCHS // 100 + 1} h on T4\n")

In [ ]:
# Restore teacher checkpoints from Drive so the training script skips them.
Path('checkpoints').mkdir(exist_ok=True)
for _name in ['teacher_r50.pth', 'teacher_r34.pth']:
    _local = Path(f'checkpoints/{_name}')
    _drive_src = Path(f'{DRIVE_BASE}/checkpoints/{_name}')
    if not _local.exists() and _drive_src.exists():
        shutil.copy2(_drive_src, _local)
        print(f'  Restored {_name} from Drive — training will be skipped.')
    elif _local.exists():
        print(f'  {_name} already present locally.')
    else:
        print(f'  {_name} not in Drive — will train from scratch.')

In [ ]:
!EPOCHS={TEACHER_EPOCHS} SEED={TEACHER_SEED} bash tools/train_teachers.sh

In [ ]:
# Sanity-check: confirm both checkpoints exist before running distillation
from pathlib import Path
import torch

for ckpt in ["checkpoints/teacher_r50.pth", "checkpoints/teacher_r34.pth"]:
    p = Path(ckpt)
    if p.exists():
        meta  = torch.load(p, map_location="cpu", weights_only=False)
        acc   = meta.get("best_acc", float("nan"))
        epoch = meta.get("epoch", "?")
        del meta
        print(f"  {ckpt}  |  epoch {epoch}  |  best_acc {acc*100:.2f}%")
    else:
        print(f"  MISSING: {ckpt}  — do not proceed until teacher training succeeds.")

In [ ]:
# Save teacher checkpoints to Drive.
_drive_ckpts = Path(f'{DRIVE_BASE}/checkpoints')
_drive_ckpts.mkdir(parents=True, exist_ok=True)
for _ckpt in ['checkpoints/teacher_r50.pth', 'checkpoints/teacher_r34.pth']:
    _src = Path(_ckpt)
    if _src.exists():
        shutil.copy2(_src, _drive_ckpts / _src.name)
        print(f'  Saved {_ckpt} to Drive.')
    else:
        print(f'  {_ckpt} not found — skipping.')

## 4 — Run distillation ablation

Runs **18 experiments** across three teacher-student pairs, two KD types, and three seeds:

| Pair | Teacher params | Student params | Capacity gap |
|------|---------------|---------------|-------------|
| R50 → R18 | 23.5 M | 11.2 M | ×2.1 |
| R34 → R18 | 21.3 M | 11.2 M | ×1.9 |
| R50 → R34 | 23.5 M | 21.3 M | ×1.1 |

Results land in `runs/{teacher}_to_{student}/{logit|feature}/seed{0,1,2}/`.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
ABLATION_EPOCHS = 100  # use 100 for publication quality; 20 for a quick smoke-test
# ─────────────────────────────────────────────────────────────────────────────

n_runs   = 18
min_per_run = 20  # rough T4 estimate for R18 student at 100 epochs
print(f"Ablation: {n_runs} runs × {ABLATION_EPOCHS} epochs")
print(f"Estimated time: ~{n_runs * min_per_run * ABLATION_EPOCHS // 100 // 60}–"
      f"{n_runs * min_per_run * ABLATION_EPOCHS // 100 // 60 + 1} h on T4")

In [ ]:
# Restore completed distillation runs from Drive so run_ablation.sh skips them.
_drive_runs = Path(f'{DRIVE_BASE}/runs')
_local_runs = Path('runs')
_local_runs.mkdir(exist_ok=True)
_n_restored = 0
if _drive_runs.exists():
    for _df in _drive_runs.rglob('*'):
        if not _df.is_file():
            continue
        _rel = _df.relative_to(_drive_runs)
        _dst = _local_runs / _rel
        if not _dst.exists():
            _dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(_df, _dst)
            _n_restored += 1
    print(f'  Restored {_n_restored} file(s) from Drive — completed runs will be skipped.')
else:
    print('  No runs/ in Drive yet — starting fresh.')

In [ ]:
!EPOCHS={ABLATION_EPOCHS} bash tools/run_ablation.sh

In [ ]:
# Sync all distillation checkpoints and TensorBoard logs to Drive.
_drive_runs = Path(f'{DRIVE_BASE}/runs')
_local_runs = Path('runs')
_n_saved = 0
for _lf in sorted(_local_runs.rglob('*')):
    if not _lf.is_file():
        continue
    _rel = _lf.relative_to(_local_runs)
    _dst = _drive_runs / _rel
    _dst.parent.mkdir(parents=True, exist_ok=True)
    if not _dst.exists() or _dst.stat().st_mtime < _lf.stat().st_mtime:
        shutil.copy2(_lf, _dst)
        _n_saved += 1
print(f'  Synced {_n_saved} file(s) to Drive under {_drive_runs}.')

## 5 — Download results

Zips `runs/` (TensorBoard logs + checkpoints) and `checkpoints/` (teacher weights)
into a single archive and triggers a browser download.

In [ ]:
from pathlib import Path
import shutil

ARCHIVE = Path('../kd_runs.zip')

print('Zipping runs/ and checkpoints/ …')
!zip -r {ARCHIVE} runs/ checkpoints/ -q

size_mb = ARCHIVE.stat().st_size / 1e6
print(f'Archive: {ARCHIVE}  ({size_mb:.0f} MB)')

# Save to Drive
_drive_archive = Path(f'{DRIVE_BASE}/kd_runs.zip')
shutil.copy2(ARCHIVE, _drive_archive)
print(f'Saved archive to Drive: {_drive_archive}')

## 6 — Results summary

Reads `best_acc` from every completed checkpoint, then prints per-seed accuracies
and computes **mean ± std** (%) across the three seeds for each (pair, KD type) cell.

Runs that have not yet completed are shown as `—` and excluded from statistics.

In [ ]:
import torch
import numpy as np
from pathlib import Path

RUNS = Path("runs")

# ── Collect best_acc from every completed checkpoint ──────────────────────────
results: dict = {}  # (pair, kd_type) -> {seed_str: float}

for ckpt_path in sorted(RUNS.glob("*/*/seed*/checkpoint_best.pth")):
    parts = ckpt_path.parts  # ('runs', pair, kd_type, 'seedN', 'checkpoint_best.pth')
    if len(parts) < 5:
        continue
    _, pair, kd_type, seed_str, _ = parts
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    acc  = float(ckpt.get("best_acc", float("nan")))
    del ckpt  # free model weights immediately
    results.setdefault((pair, kd_type), {})[seed_str] = acc

if not results:
    print("No completed runs found under runs/. Check that the ablation has finished.")
else:
    seeds = sorted({s for v in results.values() for s in v})
    W = 9  # column width

    # Header row
    seed_hdr = "".join(f"{s:>{W}}" for s in seeds)
    print(f"\n{'Pair':<22}  {'KD Type':<9}{seed_hdr}  {'Mean':>{W}}  {'±Std':>{W-2}}")
    print("─" * (22 + 9 + 2 + len(seeds) * W + W + W))

    for (pair, kd_type), seed_accs in sorted(results.items()):
        accs  = [seed_accs.get(s, float("nan")) for s in seeds]
        valid = [a for a in accs if not np.isnan(a)]
        mean  = np.mean(valid) * 100 if valid else float("nan")
        std   = np.std(valid)  * 100 if valid else float("nan")

        seed_cols = "".join(
            f"{a*100:>{W}.2f}%" if not np.isnan(a) else f"{'—':>{W}}"
            for a in accs
        )
        mean_str = f"{mean:{W}.2f}%" if not np.isnan(mean) else f"{'—':>{W}}"
        std_str  = f"{std:{W-2}.2f}%" if not np.isnan(std)  else f"{'—':>{W-2}}"
        print(f"{pair:<22}  {kd_type:<9}{seed_cols}  {mean_str}  {std_str}")

    n_found = sum(len(v) for v in results.values())
    n_total = len(results) * len(seeds)
    print(f"\n{n_found}/{n_total} seed-runs completed.")

In [ ]:
# Save the results table as a CSV to Drive.
if results:
    import csv, numpy as np
    _seeds = sorted({s for v in results.values() for s in v})
    _drive_csv = Path(f'{DRIVE_BASE}/results_summary.csv')
    with open(_drive_csv, 'w', newline='') as _f:
        _w = csv.writer(_f)
        _w.writerow(['pair', 'kd_type'] + _seeds + ['mean_pct', 'std_pct'])
        for (_pair, _kd), _seed_accs in sorted(results.items()):
            _accs  = [_seed_accs.get(s, float('nan')) for s in _seeds]
            _valid = [a for a in _accs if not np.isnan(a)]
            _mean  = np.mean(_valid) * 100 if _valid else float('nan')
            _std   = np.std(_valid)  * 100 if _valid else float('nan')
            _w.writerow(
                [_pair, _kd]
                + [f'{a*100:.4f}' if not np.isnan(a) else '' for a in _accs]
                + [f'{_mean:.4f}', f'{_std:.4f}']
            )
    print(f'Results saved to Drive: {_drive_csv}')